In [101]:
import pandas as pd
import os
import json
import numpy as np

In [102]:
def parse_result_as_df_recommended(result_file):
    with open(result_file, 'r') as f:
        lines = f.readlines()
    json_data = json.loads(('').join(lines))
    
    # Handle evaluationResult array by extracting comprehensive statistics
    if 'evaluationResult' in json_data and isinstance(json_data['evaluationResult'], list):
        eval_results = json_data['evaluationResult']
        
        # Basic statistics
        json_data['evaluationResult_count'] = len(eval_results)

        # Exact matching statistics
        exact_matches = [item for item in eval_results if item.get('doesMatchExactly', False)]
        json_data['evaluationResult_exact_matches'] = len(exact_matches) > 0
        # json_data['evaluationResult_exact_rate'] = len(exact_matches) / len(eval_results) if eval_results else 0
        
        # Syntactic matching statistics
        syntactic_matches = [item for item in eval_results if item.get('doesMatchSyntactically', False)]
        json_data['evaluationResult_syntactic_matches'] = len(syntactic_matches) > 0
        # json_data['evaluationResult_syntactic_rate'] = len(syntactic_matches) / len(eval_results) if eval_results else 0
        
        # Semantic matching statistics
        semantic_matches = [item for item in eval_results if item.get('doesMatchSemantically', False)]
        json_data['evaluationResult_semantic_matches'] = len(semantic_matches) > 0
        # json_data['evaluationResult_semantic_rate'] = len(semantic_matches) / len(eval_results) if eval_results else 0
        
        # # Top suggestions
        # json_data['evaluationResult_top_suggestion'] = eval_results[0]['suggestion'] if eval_results else None
        # json_data['evaluationResult_top_rank'] = eval_results[0]['rank'] if eval_results else None
        
        # # Average rank of matches
        # if syntactic_matches:
        #     json_data['evaluationResult_avg_syntactic_rank'] = np.mean([item['rank'] for item in syntactic_matches])
        # else:
        #     json_data['evaluationResult_avg_syntactic_rank'] = None
            
        # Remove the original array to avoid DataFrame creation issues
        del json_data['evaluationResult']

        # Delete any key that starts with 'suggestionIn' to avoid redundancy
        keys_to_delete = [key for key in json_data.keys() if key.startswith('suggestionIn')]
        for key in keys_to_delete:
            del json_data[key]
    
    # Handle any other arrays in similar fashion
    for key, value in list(json_data.items()):
        if isinstance(value, list) and key != 'evaluationResult':  # Handle other potential arrays
            json_data[f'{key}_array_length'] = len(value)
            json_data[f'{key}_as_string'] = json.dumps(value)  # Keep as JSON string if needed
            del json_data[key]  # Remove original array
    
    df = pd.DataFrame(json_data, index=[0])
    return df

In [103]:
# Updated data loading code with proper JSON array handling
FORMULA_TEST_RESULT_DIR = '../test-results/formula/multi_term/'

formula_results_df = pd.DataFrame()

print("Loading formula results...")
for root, subdirs, files in os.walk(FORMULA_TEST_RESULT_DIR):
    for file in files:
        if file.endswith('.result.json'):
            try:
                df = parse_result_as_df_recommended(os.path.join(root, file))
                formula_results_df = pd.concat([formula_results_df, df], axis=0, ignore_index=True)
            except Exception as e:
                print(f"Error processing {file}: {e}")

print(f"Formula results loaded: {len(formula_results_df)} records")

Loading formula results...
Formula results loaded: 2948 records


In [104]:
formula_results_df['suggestion_state'] = formula_results_df.apply(
    lambda row: 'no-suggestion' if not row['completionGenerated'] else row['suggestionExists'], axis=1)

In [105]:
failed_completions = formula_results_df[(formula_results_df['suggestion_state'].isin([False, 'no-suggestion']))]

In [106]:
failed_completions.groupby('modelName').size().reset_index(name='failure_count').sort_values(by='failure_count', ascending=False)

,modelName,failure_count
14,git,69
11,drone,57
20,java_meta_model,46
29,random_ski_jumping,32
23,modelo-alloy,29
28,projeto-logica,18
15,git-fixable,17
7,courses-v2,16
10,diff-and-cd2,16
5,courses-v1,16


In [107]:
from IPython.display import HTML, display
# Set CSS to make figures use more width
display(HTML("""
<style>
    .output_png {
        display: block;
        margin: 0 auto;
        max-width: 100% !important;
        width: 100% !important;
    }
    .jp-OutputArea-output {
        overflow-x: auto;
    }
</style>
"""))

# Configure matplotlib backend for better notebook integration
%matplotlib inline
import matplotlib
matplotlib.rcParams['figure.max_open_warning'] = 0

In [108]:
COLUMNS_TO_DISPLAY = ['modelName', 'line', 'character', 'incompletionLine', 'term', 'expectedCompletionWord', 'expectedCompletionLine', 'suggestion_state']

In [109]:
# Helper to render a DataFrame as an HTML table with 1-based row numbers
from IPython.display import HTML, display

def show_table(df, columns, sort_by, index_name='Row', title=None):
    view = df[columns].sort_values(sort_by).copy()
    view.index = np.arange(1, len(view) + 1)
    view.index.name = index_name
    if title:
        display(HTML(f"<h3 style='margin:0.25rem 0'>{title}</h3>"))
    display(HTML(view.to_html(index=True)))

In [110]:
# Frankervrep model Failures
failed_completion_for_model = failed_completions[(failed_completions['modelName'] == 'courses-v2-fixable')]
show_table(
    df=failed_completion_for_model,
    columns=COLUMNS_TO_DISPLAY,
    sort_by=['modelName', 'line', 'character'],
    title='Model Failures'
)

,modelName,line,character,incompletionLine,term,expectedCompletionWord,expectedCompletionLine,suggestion_state
Row,,,,,,,,
1,courses-v2-fixable,88,82,"all p : Person , disj x , y : p.projects | no ((Person <: projects).x & projects.",.,NaN,y) - p,no-suggestion
2,courses-v2-fixable,93,154,"all c : Course , p : c.projects , disj x , y : (Person <: projects).p | some c.grades[x] and some c.grades[y] implies c.grades[x] in c.grades[y].(prev +",+,NaN,iden + next),no-suggestion
3,courses-v2-fixable,93,161,"all c : Course , p : c.projects , disj x , y : (Person <: projects).p | some c.grades[x] and some c.grades[y] implies c.grades[x] in c.grades[y].(prev + iden +",+,next,next),False


In [111]:
failed_completion_for_model['line'].sort_values().unique()

array([88, 93])